In [ ]:
print("Hello World")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from git import Repo

In [ ]:
!mkdir test_repo

In [ ]:
repo_path = "test_repo/"
repo = Repo.clone_from("https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI", to_path=repo_path)

In [ ]:
loader = GenericLoader.from_filesystem(
    repo_path,
    glob = "**/*",
    suffixes=[".py"],
    parser= LanguageParser(language = Language.PYTHON, parser_threshold= 500)
)

document = loader.load()

In [ ]:
document

In [ ]:
document_splitter = RecursiveCharacterTextSplitter.from_language(
    language = Language.PYTHON,
    chunk_size = 500,
    chunk_overlap = 50
)

text_chunk = document_splitter.split_documents(document)

In [ ]:
print(f"Total chunk: {len(text_chunk)}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
def huggingface_embedding():
    embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings = huggingface_embedding()

In [ ]:
query_result = embeddings.embed_query("I am Sifat")
print(f"The result: {len(query_result)}")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [ ]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents= text_chunk,
    embedding= embeddings,
    persist_directory= "./db"
)

vector_store.persist()

In [ ]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}

)

In [ ]:
retrieved_docs = retriever.invoke("What is retriever")
retrieved_docs

In [ ]:
%pip install langchain-groq


In [ ]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature= 0,
    api_key=GROQ_API_KEY
)

In [ ]:
SYSTEM_PROMPT = """
You are a senior software engineer and code analyst.

Your role is to provide accurate, structured analysis of the provided source code context.

CONVERSATION HISTORY:
{history}

SOURCE CODE CONTEXT:
{context}

STRICT RULES:
- Do NOT guess or invent code that is not in the context.
- Do NOT provide implementation for missing code.
- If the information is not present, say "I don't know based on the provided context."
- Focus only on the provided code.

STYLE RULES:
- Be concise and clear.
- Explain responsibilities of files/modules.
- Describe data flow and key logic.
- Use bullet points, numbering, or headings when helpful.
- Keep explanations factual and professional.

Question: {question}
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)

In [ ]:
store = {}

def get_history(session_id: str):
    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()
    return store[session_id]

def build_context(question: str):
    docs = retriever.invoke(question)
    return "\n".join(doc.page_content for doc in docs)

def format_history(messages):
     if not messages:
         return "No previous conversations"
     return "\n".join(f"{m.type.capitalize()}:{m.content}" for m in messages)
 


In [ ]:
from langchain_core.runnables import RunnableLambda

base_chain = (
    {
        "question": RunnableLambda(lambda x:x["question"]),
        "context": RunnableLambda(lambda x: build_context (x["question"])),
        "history": RunnableLambda(lambda x: format_history(x["history"]))
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [ ]:
analysis_chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="question",
    history_messages_key = "history"
)

In [ ]:
response = analysis_chain.invoke(
    {"question": "What is text_splitter?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(response)